# Kronos LoRA Fine-Tuning – Google Colab

Fine-tunes **NeoQuasar/Kronos-base** on energy-sector financial data (2010–2018) using LoRA.
LoRA hyperparameters are set to the values identified as optimal in the sensitivity analysis.

| Parameter | Value | Source |
|---|---|---|
| `lora_r` | 8 | Sensitivity analysis |
| `lora_alpha` | 16 | Sensitivity analysis |
| `lora_dropout` | 0.05 | Sensitivity analysis |
| `learning_rate` | 1e-4 | Sensitivity analysis |
| `max_steps` | 1000 | Training config |

**Workflow:**
1. Setup environment
2. Fine-tune Kronos
3. Evaluate on test data (2021+)
4. Download adapter weights + results

## Setup: Clone Repository

In [ ]:
!rm -rf /content/BA
print("Cleaned up.")

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
%cd /content/BA

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

## Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm einops \
    peft transformers huggingface_hub \
    gluonts python-dotenv tiingo

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Verify Training Data

In [ ]:
from pathlib import Path

required = ["data/processed/train_data_kronos.arrow"]

all_ok = True
for p in required:
    path = Path(p)
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✅ {p}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {p}  NOT FOUND")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Missing arrow file. Push it to the repo first.")

## Fine-Tuning

Trains for 1000 steps with gradient accumulation (effective batch size = 8).
Checkpoints are saved every 250 steps to `models/kronos-lora-finetuned/`.

**Estimated time on A100: ~15–20 min**

In [ ]:
!python 02_finetuning/training/train_kronos_lora.py

In [ ]:
# Verify adapter was saved
adapter_path = Path("models/kronos-lora-finetuned/final")
if adapter_path.exists():
    files = list(adapter_path.iterdir())
    print(f"✅ Adapter saved: {adapter_path}")
    for f in files:
        print(f"   {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
else:
    print("❌ Adapter not found — check training output above")

## Evaluation on Test Data (2021+)

Runs the rolling benchmark on all energy-sector assets using the fine-tuned adapter.
Uses the optimal inference parameters from the data parameter sensitivity analysis:
`context=120, forecast=6`.

In [ ]:
!python 02_finetuning/evaluation/main_kronos_finetuned.py \
    --adapter-path models/kronos-lora-finetuned/final \
    --seed 42

## Results Summary

In [ ]:
import json
import pandas as pd
from pathlib import Path

results_path = Path("02_finetuning/results/kronos_finetuned/seed_42/final_energy_study.json")

with open(results_path) as f:
    data = json.load(f)

summary = data['summary']
rows = []
for ticker, metrics in summary.items():
    rows.append({
        'Ticker': ticker,
        'MAE': round(metrics.get('MAE_indicative', float('nan')), 4),
        'RankIC': round(metrics.get('RankIC_TimeSeries_Mean', float('nan')), 4),
        'IC': round(metrics.get('IC_Mean', float('nan')), 4),
    })

df = pd.DataFrame(rows).sort_values('RankIC', ascending=False)

print(f"Assets evaluated: {len(df)}")
print(f"Mean MAE:    {df.MAE.mean():.4f}")
print(f"Mean RankIC: {df.RankIC.mean():.4f}")
print(f"Mean IC:     {df.IC.mean():.4f}")
print()
display(df)

## Download Adapter + Results

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "kronos_lora_finetuned.zip"

with zipfile.ZipFile(zip_name, 'w') as zf:
    # LoRA adapter weights
    adapter_dir = Path("models/kronos-lora-finetuned/final")
    for f in adapter_dir.iterdir():
        zf.write(f, arcname=f"adapter/{f.name}")

    # Evaluation results
    results_dir = Path("02_finetuning/results/kronos_finetuned/seed_42")
    for f in results_dir.iterdir():
        if f.suffix == '.json':
            zf.write(f, arcname=f"results/{f.name}")

files.download(zip_name)
print(f"Downloaded: {zip_name}")